# Testing the TSFM model catalog discovery tools

A hands-on walkthrough of the **8 model-catalog discovery / read tools**. Run the cells top to
bottom and read each response: this notebook **does not assert pass/fail**, it shows you what
each tool returns so you can compare against what you expect.

| # | tool | what it does |
|---|------|--------------|
| 1 | `list_models` | browse the catalog (optionally filtered) |
| 2 | `count_models` | how many models, and a per-task breakdown |
| 3 | `list_domains` | the domains present, with counts |
| 4 | `search_models` | substring search over the catalog |
| 5 | `find_models` | a task-ranked, capability-filtered shortlist |
| 6 | `describe_candidates` | candidate models for a task (catalog order) |
| 7 | `describe_models` | compact detail for specific model ids |
| 8 | `get_model_lineage` | fine-tune + version lineage of a card |

Calls go through **MCPHub / ToolUniverse**, the same path an agent uses:

```
ToolUniverse --> stdio --> tsfm-mcp-server --> CouchStore --> CouchDB
```

Nothing is imported from `servers.tsfm` - the server runs as a separate process.

## 0. Prerequisites

From the repo root, **before** starting Jupyter:

```bash
uv run python -m ipykernel install --user \
    --name assetopsbench-mcp --display-name "assetopsbench-mcp (uv)"

docker compose -f src/couchdb/docker-compose.yaml up -d   # CouchDB on :5984

python3 src/couchdb/init_data.py --reset                  # load the catalogs

uv run jupyter lab test_model_catalog_tools.ipynb
```

`--reset` **drops** the databases first. Only the `default` scenario carries the TSFM
catalogs - `scenario_1` / `scenario_2` hold work orders only.

You can watch the data in CouchDB's web UI at <http://localhost:5984/_utils> (admin/password).

In [1]:
import json, os, sys

# Point at the repo root: adjust if this notebook is not in the repo root.
REPO = os.path.abspath(os.environ.get('AOB_REPO', '.'))
SRC  = os.path.join(REPO, 'src')
sys.path.insert(0, SRC)

# The tsfm server is spawned as a SUBPROCESS and inherits this env, so PYTHONPATH must
# be set here - sys.path only affects this notebook, not the child process.
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('COUCHDB_URL', 'http://localhost:5984')
os.environ.setdefault('COUCHDB_USERNAME', 'admin')
os.environ.setdefault('COUCHDB_PASSWORD', 'password')
os.environ['TSFM_STORE'] = 'couch'      # 'memory' to run without CouchDB

print('repo      :', REPO)
print('couchdb   :', os.environ['COUCHDB_URL'])
print('store     :', os.environ['TSFM_STORE'])

repo      : /Users/miao/Asset/AssetOpsBench
couchdb   : http://localhost:5984
store     : couch


### Is CouchDB up and seeded?

Expect `model_catalog` to exist. On a fresh `--reset` it holds **1** card (`ttm_96_28`).

In [2]:
import urllib.request, base64

def couch(path):
    url = os.environ['COUCHDB_URL'].rstrip('/') + path
    req = urllib.request.Request(url)
    tok = base64.b64encode(f"{os.environ['COUCHDB_USERNAME']}:{os.environ['COUCHDB_PASSWORD']}".encode()).decode()
    req.add_header('Authorization', 'Basic ' + tok)
    return json.load(urllib.request.urlopen(req, timeout=10))

try:
    print('databases    :', couch('/_all_dbs'))
    print('model_catalog:', couch('/model_catalog')['doc_count'], 'docs')
except Exception as e:
    print('CouchDB not reachable ->', e)
    print('Start it, or set TSFM_STORE=memory in the cell above.')

databases    : ['_replicator', '_users', 'asset', 'catalog', 'failure_code', 'failure_mode', 'feature_catalog', 'iot', 'model_catalog', 'vibration', 'workorder']
model_catalog: 5 docs


## 1. Connect through MCPHub

`load_tools` spawns the tsfm server as a subprocess and discovers its tools over stdio.

In [3]:
from mcphub import ToolUniverse

# 'uv run tsfm-mcp-server' is the normal launcher; this fallback needs no uv.
SERVER_CMD = os.environ.get('SERVER_CMD', f'{sys.executable} -m servers.tsfm.main').split()

tu = ToolUniverse(servers={'tsfm': SERVER_CMD})
n = tu.load_tools(servers=['tsfm'])
print(f'{n} tools discovered\n')
print('\n'.join(sorted(tu.all_tools)))

20 tools discovered

tsfm.characterize_series
tsfm.count_models
tsfm.data_quality
tsfm.deprecate_model
tsfm.describe_candidates
tsfm.describe_models
tsfm.find_models
tsfm.get_model_lineage
tsfm.hf_stats
tsfm.list_domains
tsfm.list_models
tsfm.list_tasks
tsfm.model_template
tsfm.new_model_version
tsfm.profile_series
tsfm.register_finetuned
tsfm.register_model
tsfm.resolve_model
tsfm.search_models
tsfm.update_model


## 1. `list_models` - browse the catalog

The broad entry point: full cards, unranked, with no filter by default (only `active` cards are
returned). Pair it with `describe_models` for by-id detail, or `find_models` / `search_models`
when you need ranking or text search.

**Expect:** every active card. We keep the ids around for the later cells.

In [4]:
r = tu.run({'name': 'tsfm.list_models', 'arguments': {}})
models = r.get('result', r).get('models', [])
print(f'{len(models)} active model(s):\n')
for m in sorted(models, key=lambda x: x['model_id']):
    print(f"  {m['model_id']:<20} {m.get('status'):<10} {m.get('task_ids')}")

# keep the ids for later cells (describe_models / get_model_lineage)
ALL_IDS = [m['model_id'] for m in models]

# filtered example: only forecasting cards
r = tu.run({'name': 'tsfm.list_models', 'arguments': {'task_id': 'tsfm_forecasting'}})
fc = r.get('result', r).get('models', [])
print(f'\nfiltered task_id=tsfm_forecasting -> {len(fc)} model(s)')

1 active model(s):

  ttm_96_28            active     ['tsfm_forecasting']

filtered task_id=tsfm_forecasting -> 1 model(s)


## 2. `count_models` - how many, by task

A catalog-size summary over active cards: the `total`, plus a per-task breakdown (a card that
lists several `task_ids` is counted under each). Takes no arguments.

**Expect:** `total` and a `by_task` mapping.

In [5]:
r = tu.run({'name': 'tsfm.count_models', 'arguments': {}})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "total": 1,
    "by_task": {
      "tsfm_forecasting": 1
    }
  }
}


## 3. `list_domains` - the domains present

Enumerates the values you can pass to the `domain` filter of `list_models` / `find_models` /
`describe_candidates`, each with its model count (cards without a domain are grouped under
`unspecified`). Optionally scoped to a single task.

**Expect:** a domain -> count mapping.

In [6]:
r = tu.run({'name': 'tsfm.list_domains', 'arguments': {}})
print('# all domains')
print(json.dumps(r, indent=2, default=str))

r = tu.run({'name': 'tsfm.list_domains', 'arguments': {'task_id': 'tsfm_forecasting'}})
print('\n# scoped to tsfm_forecasting')
print(json.dumps(r, indent=2, default=str))

# all domains
{
  "result": {
    "domains": {
      "general": 1
    }
  }
}

# scoped to tsfm_forecasting
{
  "result": {
    "domains": {
      "general": 1
    }
  }
}


## 4. `search_models` - substring search

Case-insensitive substring match over each card's `model_id`, `description`, `model_family` and
`tags`. Use it when you have a keyword in mind (a family, vendor, or capability). `text` is
required; empty text is rejected.

**Expect:** every card whose fields contain the substring.

In [7]:
r = tu.run({'name': 'tsfm.search_models', 'arguments': {'text': 'ttm'}})
hits = r.get('result', r).get('models', [])
print("text='ttm' ->", len(hits), 'hit(s):', [m['model_id'] for m in hits])

# empty text is rejected
r = tu.run({'name': 'tsfm.search_models', 'arguments': {'text': ''}})
print('\nempty text ->', r.get('result', r))

text='ttm' -> 1 hit(s): ['ttm_96_28']

empty text -> {'error': 'text is required: a substring to search for (use list_models to browse all)'}


## 5. `find_models` - task-ranked shortlist

Narrows to cards that support `task_id`, applies the optional capability filters, and returns at
most `top_k`. A card lacking a filtered field is dropped (e.g. classical models have no
`context_length`, so `min_context_length` excludes them).

**Expect:** a ranked shortlist for the task; the capability filter shrinks it.

In [8]:
r = tu.run({'name': 'tsfm.find_models',
            'arguments': {'task_id': 'tsfm_forecasting', 'top_k': 5}})
top = r.get('result', r).get('models', [])
print('task=tsfm_forecasting, top_k=5 ->', [m['model_id'] for m in top])

# add a capability filter
r = tu.run({'name': 'tsfm.find_models',
            'arguments': {'task_id': 'tsfm_forecasting', 'min_context_length': 512, 'top_k': 5}})
top2 = r.get('result', r).get('models', [])
print('min_context_length=512 ->', [m['model_id'] for m in top2])

task=tsfm_forecasting, top_k=5 -> ['ttm_96_28']
min_context_length=512 -> []


## 6. `describe_candidates` - candidates for a task (catalog order)

A HuggingGPT-style shortlist: the cards that support `task_id`, capped at `top_k`, presented in
**catalog order** (no ranking or scoring - you decide which to use). Follow up with `hf_stats`
to judge popularity yourself.

**Expect:** `task_id` echoed and a `candidates` list.

In [9]:
r = tu.run({'name': 'tsfm.describe_candidates',
            'arguments': {'task_id': 'tsfm_forecasting', 'top_k': 5}})
res = r.get('result', r)
print('task_id   :', res.get('task_id'))
print('candidates:', [c.get('model_id') for c in res.get('candidates', [])])

task_id   : tsfm_forecasting
candidates: ['ttm_96_28']


## 7. `describe_models` - detail for specific ids

The by-id detail lookup that pairs with `list_models` / `find_models`: a compact record per id
(description, family, `sktime_class`, `context_length`, domain, tags) rather than the full card.
Ids not in the catalog are returned in a separate `unknown` list rather than raising an error.

**Expect:** compact records for the known ids, plus a bogus id surfaced under `unknown`.

In [10]:
probe = ALL_IDS[:2] + ['no_such_model']     # reuse ids from section 1, add a bogus one
r = tu.run({'name': 'tsfm.describe_models', 'arguments': {'model_ids': probe}})
print(json.dumps(r, indent=2, default=str))

{
  "result": {
    "models": [
      {
        "model_id": "ttm_96_28",
        "description": "Pretrained TinyTimeMixer forecasting model, context length 96, prediction horizon 28. General-purpose zero-shot multivariate forecaster; good for short-context, short-horizon tasks.",
        "family": "TinyTimeMixer",
        "sktime_class": "sktime.forecasting.ttm.TinyTimeMixerForecaster",
        "context_length": 96,
        "domain": "general",
        "tags": [
          "ttm",
          "forecasting",
          "zero-shot",
          "short-context"
        ]
      }
    ],
    "unknown": [
      "no_such_model"
    ],
    "message": "described 1 model(s); unknown: ['no_such_model']."
  }
}


## 8. `get_model_lineage` - lineage of a card

Traces a card's fine-tune chain (base-model ancestors + fine-tuned descendants, from
`register_finetuned`) and its version links (`supersedes` / `superseded_by`, from
`new_model_version`). Use it to see where a card came from and what replaced it.

**Expect:** the lineage graph for the chosen id.

In [11]:
mid = ALL_IDS[0] if ALL_IDS else 'ttm_96_28'   # any existing id from section 1
r = tu.run({'name': 'tsfm.get_model_lineage', 'arguments': {'model_id': mid}})
print(f'lineage for {mid}:')
print(json.dumps(r, indent=2, default=str))

lineage for ttm_96_28:
{
  "result": {
    "model_id": "ttm_96_28",
    "ancestors": [],
    "root": "ttm_96_28",
    "descendants": [],
    "supersedes": null,
    "superseded_by": null
  }
}


## Clean up

Close the stdio session. These eight tools are read-only, so nothing in the catalog was changed.

In [12]:
tu.close()
print('closed')

closed
